# Supervised Fine-Tuning (SFT) with PEFT of Amazon Nova 2.0 Lite using Amazon SageMaker Training Job

## Lab 1 - Data preparation

In this notebook, we are going to prepare the dataset for later on fine-tune Amazon Nova Lite 2.0

***

In [ ]:
%pip install -r requirements.txt

## Prerequisites

If you are going to use Sagemaker in a local environment. You need access to an IAM Role with the required permissions for Sagemaker. You can find [here](https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-roles.html) more about it.

In [ ]:
import sagemaker
import boto3

sess = sagemaker.Session()
sagemaker_session_bucket = None

if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = sagemaker.get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

sess = sagemaker.Session(default_bucket=sagemaker_session_bucket)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {sess.default_bucket()}")
print(f"sagemaker session region: {sess.boto_region_name}")

***

## Prepare the dataset

We are going to load [HuggingFaceH4/Multilingual-Thinking](https://huggingface.co/datasets/HuggingFaceH4/Multilingual-Thinking) dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "HuggingFaceH4/Multilingual-Thinking", split="train"
)

dataset

In [ ]:
import pandas as pd

df = pd.DataFrame(dataset)

df.head()

In [ ]:
from sklearn.model_selection import train_test_split

_, val = train_test_split(df, test_size=0.2, random_state=42)
train, test = train_test_split(_, test_size=10, random_state=42)

print("Number of train elements: ", len(train))
print("Number of val elements: ", len(val))
print("Number of test elements: ", len(test))

### Utility functions

In [ ]:
import tiktoken
import json

model_id = "o200k_base"

def count_tokens(text):
    """Count number of tokens in a string"""
    encoding = tiktoken.get_encoding(model_id)
    return len(encoding.encode(text))


def count_message_tokens(messages_dict):
    """Count total tokens in the entire message structure"""
    message_json = json.dumps(messages_dict, ensure_ascii=False)
    return count_tokens(message_json)


def truncate_entire_message(messages, max_tokens=30000):
    """Truncate the entire message structure if it's still too long"""
    current_tokens = count_message_tokens(messages)

    if current_tokens <= max_tokens:
        return messages

    think_limit = max(500, max_tokens // 8)
    content_limit = max(1000, max_tokens // 4)

    for message in messages.get("messages", []):
        if message.get("role") == "assistant":
            for content in message.get("content", []):
                if "reasoningContent" in content:
                    reasoning_text = content["reasoningContent"]["reasoningText"]["text"]
                    truncated = truncate_message(reasoning_text, max_tokens=think_limit)
                    content["reasoningContent"]["reasoningText"]["text"] = truncated
                elif "text" in content:
                    content["text"] = truncate_message(
                        content["text"], max_tokens=content_limit
                    )

        current_tokens = count_message_tokens(messages)
        if current_tokens <= max_tokens:
            break

    return messages


def truncate_message(message_str, max_tokens=20000):
    """Truncate the text based on token count, not character count"""
    current_tokens = count_tokens(message_str)

    if current_tokens <= max_tokens:
        return message_str

    print(f"Truncating message from {current_tokens} to ~{max_tokens} tokens...")

    encoding = tiktoken.get_encoding(model_id)
    tokens = encoding.encode(message_str)
    truncated_tokens = tokens[: max_tokens - 3]
    truncated_text = encoding.decode(truncated_tokens) + "..."

    return truncated_text

Let's format the dataset using the Nova 2.0 Converse format. Nova 2.0 introduces the `reasoningContent` field, which replaces the `<think>` tags used in Nova 1.0 with a proper structured field for chain-of-thought reasoning:

```
{
    "system": [{"text": Content of the System prompt}],
    "messages": [
        {
            "role": "user",
            "content": [{"text": Content of the user prompt}]
        },
        {
            "role": "assistant",
            "content": [
                {
                    "reasoningContent": {
                        "reasoningText": {"text": Reasoning / thinking content}
                    }
                },
                {"text": Content of the final answer}
            ]
        }
    ]
}
```

By adding `reasoningContent`, models will train better than those without reasoning content.

In [ ]:
import textwrap

max_tokens = 3072
max_tokens_think = 2250

def prepare_dataset(sample):
    """Prepare dataset in the required Nova 2.0 Converse format with reasoningContent"""
    messages = {"system": [], "messages": []}

    for el in sample["messages"]:
        if el["role"] == "system":
            system_prompt = """
            You are an AI assistant that reasons in {language} and responds in English.

            Always reason through the problem in {language}, then provide your final answer in English.
            """

            system_prompt = system_prompt.format(language=sample["reasoning_language"])
            system_prompt = textwrap.dedent(system_prompt).strip()

            messages["system"].append({"text": system_prompt.lower()})
        elif el["role"] == "user":
            messages["messages"].append(
                {"role": "user", "content": [{"text": el["content"].lower()}]}
            )
        else:
            if (
                el["thinking"] is not None
                and el["thinking"] != ""
                and el["thinking"] != "null"
            ):
                reasoning_text = truncate_message(
                    el["thinking"].lower(), max_tokens=max_tokens_think
                )
                messages["messages"].append(
                    {
                        "role": "assistant",
                        "content": [
                            {
                                "reasoningContent": {
                                    "reasoningText": {"text": reasoning_text}
                                }
                            },
                            {"text": el["content"].lower()},
                        ],
                    }
                )
            else:
                content_text = truncate_message(
                    el["content"].lower(), max_tokens=max_tokens
                )
                messages["messages"].append(
                    {
                        "role": "assistant",
                        "content": [
                            {"text": content_text},
                        ],
                    }
                )

    total_tokens = count_message_tokens(messages)
    if total_tokens > max_tokens:
        print(
            f"Warning: Total message tokens ({total_tokens}) still too high, applying additional truncation..."
        )
        messages = truncate_entire_message(messages, max_tokens=max_tokens)

    return messages

In [ ]:
def prepare_dataset_test(sample):
    """Parse sample and format it for test dataset."""
    message = {
        "system": "",
        "query": "",
        "response": "",
    }

    for el in sample["messages"]:
        if el["role"] == "system":
            system_prompt = """
            You are an AI assistant that reasons in {language} and responds in English.

            Always reason through the problem in {language}, then provide your final answer in English.
            """

            system_prompt = system_prompt.format(language=sample["reasoning_language"])
            system_prompt = textwrap.dedent(system_prompt).strip()

            message["system"] = system_prompt.lower()
        elif el["role"] == "user":
            message["query"] = el["content"].lower()
        else:
            if (
                el["thinking"] is not None
                and el["thinking"] != ""
                and el["thinking"] != "null"
            ):
                reasoning_text = truncate_message(
                    el["thinking"].lower(), max_tokens=max_tokens_think
                )
                message["response"] = (
                    f"{reasoning_text}\n\n{el['content']}".lower()
                )
            else:
                message["response"] = truncate_message(
                    el["content"].lower(), max_tokens=max_tokens
                )

    return message

In [ ]:
import json
from random import randint


def has_empty_text(sample):
    """Filter out samples with empty text content in assistant messages"""
    for msg in sample.get("messages", []):
        for content in msg.get("content", []):
            if "text" in content and (content["text"] is None or content["text"].strip() == ""):
                return False
    return True


# Process training data
train_data = []
train_filtered = 0
for _, row in train.iterrows():
    sample = row.to_dict()
    formatted = prepare_dataset(sample)
    if has_empty_text(formatted):
        train_data.append(formatted)
    else:
        train_filtered += 1

print(f"Training: filtered {train_filtered} samples with empty text content")
print(f"Final training samples: {len(train_data)}")

# Process validation data
val_data = []
val_filtered = 0
for _, row in val.iterrows():
    sample = row.to_dict()
    formatted = prepare_dataset(sample)
    if has_empty_text(formatted):
        val_data.append(formatted)
    else:
        val_filtered += 1

print(f"Validation: filtered {val_filtered} samples with empty text content")
print(f"Final validation samples: {len(val_data)}")

print(json.dumps(train_data[randint(0, len(train_data) - 1)], indent=2, ensure_ascii=False))

The test dataset will be formatted with the structure below:

```
{
    "system": "Optional - String containing the system prompt",
    "query": "String containing the input prompt",
    "response": "String containing the expected model output"
}
```

In [ ]:
test_data = []
for _, row in test.iterrows():
    sample = row.to_dict()
    formatted = prepare_dataset_test(sample)
    test_data.append(formatted)

print(json.dumps(test_data[randint(0, len(test_data) - 1)], indent=2, ensure_ascii=False))

### Upload to Amazon S3

In [ ]:
s3_client = boto3.client("s3")

if default_prefix:
    input_path = f"{default_prefix}/datasets/nova-2-lite-sft-peft"
else:
    input_path = f"datasets/nova-2-lite-sft-peft"

train_dataset_s3_path = f"s3://{bucket_name}/{input_path}/train/dataset.jsonl"
val_dataset_s3_path = f"s3://{bucket_name}/{input_path}/val/dataset.jsonl"
test_dataset_s3_path = f"s3://{bucket_name}/{input_path}/test/gen_qa.jsonl"

In [ ]:
import os

os.makedirs("./data/train", exist_ok=True)
os.makedirs("./data/val", exist_ok=True)
os.makedirs("./data/test", exist_ok=True)

with open("./data/train/dataset.jsonl", "w") as f:
    for sample in train_data:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

with open("./data/val/dataset.jsonl", "w") as f:
    for sample in val_data:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

with open("./data/test/gen_qa.jsonl", "w") as f:
    for sample in test_data:
        f.write(json.dumps(sample, ensure_ascii=False) + "\n")

s3_client.upload_file(
    "./data/train/dataset.jsonl", bucket_name, f"{input_path}/train/dataset.jsonl"
)

s3_client.upload_file(
    "./data/val/dataset.jsonl", bucket_name, f"{input_path}/val/dataset.jsonl"
)

s3_client.upload_file(
    "./data/test/gen_qa.jsonl", bucket_name, f"{input_path}/test/gen_qa.jsonl"
)

print(f"Training data uploaded to:")
print(train_dataset_s3_path)
print(val_dataset_s3_path)
print(test_dataset_s3_path)